In [ ]:
# ===== 0. Cấu hình (sửa ở đây rồi chạy lần lượt 1 -> 7)
RUN_MODE = "probe"            # "precut" (runtime CPU là đủ) | "probe" | "train" | "eval"
BRANCH = "htstep_adapter"
REPO_URL = "https://github.com/im-xiaoming/Omni-Retriever-Custom.git"
REPO = "/content/Omni-fix"
WORK = "/content/htstep"      # model, clip, run (đĩa local của Colab)
HF_CLIPS = ""                 # repo HF dataset RIÊNG của nhóm (private) cho clip, vd "<nhóm>/htstep-omni-clips"
HF_RUN_REPO = ""              # repo HF model RIÊNG của nhóm (private) cho checkpoint tốt nhất + metrics.json
GRAD_CACHE_CHUNK = ""         # train: K chọn từ bảng probe
GRADIENT_CHECKPOINTING = "True"
EVAL_EVERY_FRAC = "0.25"      # 0.5 nếu probe cho thấy một lần eval val > 15 % một epoch
PROBE_UNIT = "1000"           # tar của htstep-videos dùng cho probe
PRECUT_UNITS = ""             # "" = cắt tất cả; "1000" = chỉ tar của probe (cắt trước, phần còn lại cắt sau)
PRECUT_JOBS = "4"
DRIVE_OUT = "/content/drive/MyDrive/Colab Notebooks/code KL/Omni/htstep_lora"
BENCH_ROOT = "/content/drive/MyDrive/Colab Notebooks/code KL/Omni_benchmark_data"
BENCH_MANIFEST = f"{BENCH_ROOT}/benchmark/manifests/omniretriever_bench_final.jsonl"
BENCH_MEDIA_ROOT = BENCH_ROOT  # thư mục CHỨA benchmark/ (record ghi "benchmark/videos/...")

import os, subprocess
from google.colab import userdata
assert RUN_MODE in ("precut", "probe", "train", "eval"), RUN_MODE
assert HF_CLIPS, "đặt HF_CLIPS (repo riêng của nhóm)"
os.environ.update({
    "HF_TOKEN": userdata.get("HF_TOKEN"), "RUN_MODE": RUN_MODE, "WORK": WORK, "HF_CLIPS": HF_CLIPS,
    "HF_RUN_REPO": HF_RUN_REPO, "GRAD_CACHE_CHUNK": GRAD_CACHE_CHUNK, "GRADIENT_CHECKPOINTING": GRADIENT_CHECKPOINTING,
    "EVAL_EVERY_FRAC": EVAL_EVERY_FRAC, "PROBE_UNIT": PROBE_UNIT, "PRECUT_UNITS": PRECUT_UNITS,
    "PRECUT_JOBS": PRECUT_JOBS, "BENCH_MANIFEST": BENCH_MANIFEST, "BENCH_MEDIA_ROOT": BENCH_MEDIA_ROOT,
})

def sh(stages):
    """One stage of scripts/run_vast.sh (the same commands the Vast machine runs over SSH)."""
    ip = get_ipython()
    ip.system(f'cd "{REPO}" && STAGES="{stages}" bash scripts/run_vast.sh')
    if ip.user_ns.get("_exit_code", 0) != 0:
        raise SystemExit(f"run_vast.sh [{stages}] dừng (exit {ip.user_ns.get('_exit_code')}): xem log ở {WORK}/logs")
print("RUN_MODE", RUN_MODE)

In [ ]:
# ===== 1. Code + thư viện + GPU
from google.colab import drive
drive.mount("/content/drive")
if not os.path.isdir(REPO):
    !git clone -q -b {BRANCH} {REPO_URL} {REPO}
!cd {REPO} && git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git pull -q && git log -1 --format="commit %H %s"
# cài thư viện theo training/requirements.txt (precut: chỉ transformers 4.51.3 + decord + av);
# GPU dưới 20GB hoặc không có bf16 (T4): in "không nạp được bf16, dùng L4 / A100" và dừng (trừ precut)
sh("install")

In [ ]:
# ===== 2. Model + adapter (kiểm sha256 với YunzeLiu/OmniRetriever-7B), vá RoPE
sh("models")

In [ ]:
# ===== 3. Clip: precut = cắt + upload lên HF_CLIPS; các mode khác = tải shard clip cần dùng
sh("clips")
if RUN_MODE == "precut":
    !cat {WORK}/precut/precut_summary.json

In [ ]:
# ===== 4. Probe VRAM / tốc độ (mục 6 của plan)
if RUN_MODE == "probe":
    sh("run")
    os.makedirs(f"{DRIVE_OUT}/probe", exist_ok=True)
    !cp {WORK}/probe/vram_probe_*.json "{DRIVE_OUT}/probe/" && ls -la "{DRIVE_OUT}/probe"

In [ ]:
# ===== 5. Train (kiểm protocol trước; resume nếu đã có checkpoint)
if RUN_MODE == "train":
    assert GRAD_CACHE_CHUNK, "đặt GRAD_CACHE_CHUNK theo bảng probe"
    sh("run")

In [ ]:
# ===== 6. Eval: OmniRetriever-Bench (pt + checkpoint tốt nhất) + export_step_emb cho UniAV
if RUN_MODE == "eval":
    sh("run")

In [ ]:
# ===== 7. Tóm tắt, chỗ lưu file
import glob, json, shutil
os.makedirs(DRIVE_OUT, exist_ok=True)
for pat in ("probe/vram_probe_*.json", "precut/precut_summary.json", "runs/*/metrics.json", "bench/bench_*.json",
            "bench/metrics_*.json", "omni_step_emb_*.npz", "logs/*.log"):
    for f in glob.glob(f"{WORK}/{pat}"):
        dst = os.path.join(DRIVE_OUT, os.path.relpath(f, WORK))
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy(f, dst)
        print(dst)
for f in glob.glob(f"{WORK}/runs/*/metrics.json"):
    m = json.load(open(f))
    print(f, "best t2v R@1", m.get("best"), "@ step", m.get("best_step"))
print("HF:", HF_CLIPS, HF_RUN_REPO)